# CAFA5 Medium Graph Training Results

Objective: run a medium-scale graph training job with the upgraded Savio script, then plot metrics from the saved result files.

Default medium run:
- `BPO`, `CCO`, `MFO`
- `4096 / 1024 / 1024` train/val/test proteins per aspect, capped by the available full split size
- `5` epochs
- `savio2_1080ti`, 4 GPUs via `scripts/savio_train_full_graph.sh`
- metrics: loss, micro-F1, macro-F1, Fmax, precision, recall

This notebook intentionally uses a medium split directory instead of the full split directory. It is meant to validate the upgraded training script and produce plots before launching the full run.

In [ ]:
# Setup
from __future__ import annotations

import json
import os
import re
import shlex
import subprocess
import sys
from datetime import datetime
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

from IPython.display import display

VALID_ASPECTS = {'BPO', 'CCO', 'MFO'}


def find_repo_root(start: Path) -> tuple[Path, str]:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate, 'cwd_search'
    return start, 'cwd_fallback'


def resolve_repo_root() -> tuple[Path, str]:
    configured = os.environ.get('CAFA5_REPO_ROOT')
    if configured:
        return Path(configured).expanduser().resolve(), 'env:CAFA5_REPO_ROOT'
    return find_repo_root(Path.cwd())


def parse_aspects(value: str) -> list[str]:
    parsed = [part.strip().upper() for part in value.replace(',', ' ').split() if part.strip()]
    invalid = [aspect for aspect in parsed if aspect not in VALID_ASPECTS]
    if invalid:
        raise ValueError(f'Invalid aspects: {invalid}; expected subset of {sorted(VALID_ASPECTS)}')
    if not parsed:
        raise ValueError('At least one aspect is required')
    return parsed


def env_bool(name: str, default: str = '0') -> bool:
    return os.environ.get(name, default).strip().lower() in {'1', 'true', 'yes', 'y'}


REPO_ROOT, REPO_ROOT_SOURCE = resolve_repo_root()
SERVER_USER = os.environ.get('USER', 'bensonli')
RUN_ROOT = Path(os.environ.get('CAFA5_RUN_ROOT', f'/global/scratch/users/{SERVER_USER}/cafa5_outputs'))
GRAPH_CACHE_DIR = RUN_ROOT / 'graph_cache'
FULL_SPLIT_DIR = GRAPH_CACHE_DIR / 'splits'
TRAIN_SCRIPT = REPO_ROOT / 'scripts' / 'savio_train_full_graph.sh'
PYTHON_BIN = Path(os.environ.get('PYTHON_BIN', f'/global/home/users/{SERVER_USER}/venvs/cafa5-notebook/bin/python'))

NOTEBOOK_RUN_ID = os.environ.get('CAFA5_MEDIUM_RUN_ID') or globals().get('NOTEBOOK_RUN_ID') or datetime.now().strftime('%Y%m%d_%H%M%S')
FRAMEWORK = os.environ.get('CAFA5_TRAIN_FRAMEWORK', 'pyg')
ASPECTS = parse_aspects(os.environ.get('CAFA5_MEDIUM_ASPECTS', 'BPO CCO MFO'))
MIN_TERM_FREQUENCY = int(os.environ.get('CAFA5_MIN_TERM_FREQUENCY', '20'))
SEED = int(os.environ.get('CAFA5_TRAIN_SEED', '2026'))

MEDIUM_TRAIN_LIMIT = int(os.environ.get('CAFA5_MEDIUM_TRAIN_LIMIT', '4096'))
MEDIUM_VAL_LIMIT = int(os.environ.get('CAFA5_MEDIUM_VAL_LIMIT', '1024'))
MEDIUM_TEST_LIMIT = int(os.environ.get('CAFA5_MEDIUM_TEST_LIMIT', '1024'))
MEDIUM_EPOCHS = int(os.environ.get('CAFA5_MEDIUM_EPOCHS', '5'))
MEDIUM_BATCH_SIZE = int(os.environ.get('CAFA5_MEDIUM_BATCH_SIZE', '8'))
MEDIUM_NUM_WORKERS = int(os.environ.get('CAFA5_MEDIUM_NUM_WORKERS', '2'))
MEDIUM_HIDDEN_DIM = int(os.environ.get('CAFA5_MEDIUM_HIDDEN_DIM', '128'))

METRIC_THRESHOLD = os.environ.get('METRIC_THRESHOLD', '0.5')
FMAX_THRESHOLD_STEP = os.environ.get('FMAX_THRESHOLD_STEP', '0.01')
PROGRESS_MODE = os.environ.get('PROGRESS_MODE', 'log')
PROGRESS_EVERY = os.environ.get('PROGRESS_EVERY', '10')
DEVICE = os.environ.get('DEVICE', 'auto')
RUN_NAME = os.environ.get('CAFA5_MEDIUM_RUN_NAME', f'medium_graph_{FRAMEWORK}_mtf{MIN_TERM_FREQUENCY}_{NOTEBOOK_RUN_ID}')
MEDIUM_SPLIT_DIR = GRAPH_CACHE_DIR / f'splits_medium_t{MEDIUM_TRAIN_LIMIT}_v{MEDIUM_VAL_LIMIT}_s{MEDIUM_TEST_LIMIT}_mtf{MIN_TERM_FREQUENCY}'
RUN_DIR = GRAPH_CACHE_DIR / 'training_runs' / RUN_NAME
IA_FILE = Path(os.environ.get('IA_FILE', str(Path.home() / 'cafa5' / 'IA.txt'))).expanduser()
GO_OBO_FILE = Path(os.environ.get('GO_OBO_FILE', str(Path.home() / 'cafa5' / 'go-basic.obo'))).expanduser()
GROUND_TRUTH_FILE = Path(os.environ.get('GROUND_TRUTH_FILE', str(Path.home() / 'cafa5' / 'test_terms.tsv'))).expanduser()
SUBMIT_JOB = env_bool('CAFA5_SUBMIT_MEDIUM_TRAINING', '1')
DRY_RUN = env_bool('CAFA5_DRY_RUN_MEDIUM_TRAINING', '0')

config = {
    'repo_root': str(REPO_ROOT),
    'repo_root_source': REPO_ROOT_SOURCE,
    'run_root': str(RUN_ROOT),
    'graph_cache_dir': str(GRAPH_CACHE_DIR),
    'full_split_dir': str(FULL_SPLIT_DIR),
    'medium_split_dir': str(MEDIUM_SPLIT_DIR),
    'train_script': str(TRAIN_SCRIPT),
    'python_bin': str(PYTHON_BIN),
    'run_name': RUN_NAME,
    'run_dir': str(RUN_DIR),
    'aspects': ASPECTS,
    'limits': {'train': MEDIUM_TRAIN_LIMIT, 'val': MEDIUM_VAL_LIMIT, 'test': MEDIUM_TEST_LIMIT},
    'epochs': MEDIUM_EPOCHS,
    'batch_size': MEDIUM_BATCH_SIZE,
    'num_workers': MEDIUM_NUM_WORKERS,
    'hidden_dim': MEDIUM_HIDDEN_DIM,
    'metric_threshold': METRIC_THRESHOLD,
    'fmax_threshold_step': FMAX_THRESHOLD_STEP,
    'progress_mode': PROGRESS_MODE,
    'progress_every': PROGRESS_EVERY,
    'device': DEVICE,
    'ia_file': str(IA_FILE),
    'go_obo_file': str(GO_OBO_FILE),
    'ground_truth_file': str(GROUND_TRUTH_FILE),
    'submit_job': SUBMIT_JOB,
    'dry_run': DRY_RUN,
}
print(json.dumps(config, indent=2))

required_repo_files = [TRAIN_SCRIPT, REPO_ROOT / 'train_minimal_graph_model.py']
missing_repo_files = [str(path) for path in required_repo_files if not path.exists()]
if missing_repo_files:
    raise FileNotFoundError(f'Missing repo files: {missing_repo_files}')

## Build Medium Splits

This cell mirrors the existing full graph split manifests into a smaller medium split directory. It does not overwrite the full split under `graph_cache/splits`.

In [ ]:
# Create a medium split from the existing full graph split manifests.
def read_ids(path: Path) -> list[str]:
    return [line.strip() for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def write_lines(path: Path, values: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{value}\n' for value in values), encoding='utf-8')
    print(f'[write] {path} ({len(values)} lines)')


if not FULL_SPLIT_DIR.exists():
    raise FileNotFoundError(f'Missing full split directory: {FULL_SPLIT_DIR}')
if not (FULL_SPLIT_DIR / 'summary.json').exists():
    raise FileNotFoundError(f'Missing full split summary: {FULL_SPLIT_DIR / "summary.json"}')

limits = {'train': MEDIUM_TRAIN_LIMIT, 'val': MEDIUM_VAL_LIMIT, 'test': MEDIUM_TEST_LIMIT}
medium_summary = {
    'source_split_dir': str(FULL_SPLIT_DIR),
    'root': str(MEDIUM_SPLIT_DIR),
    'run_name': RUN_NAME,
    'min_term_frequency': MIN_TERM_FREQUENCY,
    'seed': SEED,
    'limits': limits,
    'aspects': {},
}

rows = []
for aspect in ASPECTS:
    source_aspect_dir = FULL_SPLIT_DIR / aspect.lower()
    target_aspect_dir = MEDIUM_SPLIT_DIR / aspect.lower()
    if not source_aspect_dir.exists():
        raise FileNotFoundError(f'Missing source aspect split directory: {source_aspect_dir}')
    medium_summary['aspects'][aspect] = {'source': str(source_aspect_dir), 'counts': {}, 'source_counts': {}}
    for split_name, limit in limits.items():
        source_path = source_aspect_dir / f'{split_name}.txt'
        if not source_path.exists():
            raise FileNotFoundError(f'Missing split file: {source_path}')
        ids = read_ids(source_path)
        subset = ids[:min(limit, len(ids))]
        write_lines(target_aspect_dir / f'{split_name}.txt', subset)
        medium_summary['aspects'][aspect]['counts'][split_name] = len(subset)
        medium_summary['aspects'][aspect]['source_counts'][split_name] = len(ids)
        rows.append({'aspect': aspect, 'split': split_name, 'selected': len(subset), 'source_total': len(ids)})
    source_summary_path = source_aspect_dir / 'summary.json'
    if source_summary_path.exists():
        (target_aspect_dir / 'summary.json').write_text(source_summary_path.read_text(encoding='utf-8'), encoding='utf-8')

MEDIUM_SPLIT_DIR.mkdir(parents=True, exist_ok=True)
(MEDIUM_SPLIT_DIR / 'summary.json').write_text(json.dumps(medium_summary, indent=2), encoding='utf-8')
print(f'[write] {MEDIUM_SPLIT_DIR / "summary.json"}')

if pd is not None:
    display(pd.DataFrame(rows))
else:
    print(json.dumps(rows, indent=2))

## Submit the Savio Job

This cell submits `scripts/savio_train_full_graph.sh` with medium-run environment variables. The script still uses the upgraded training code: 4 GPU allocation, per-aspect GPU assignment, progress logs, and expanded metrics.

In [ ]:
# Submit the medium training job to Savio.
def format_cmd(cmd: list[object]) -> str:
    return ' '.join(shlex.quote(str(part)) for part in cmd)


submit_env = os.environ.copy()
submit_env.update(
    {
        'REPO_ROOT': str(REPO_ROOT),
        'RUN_ROOT': str(RUN_ROOT),
        'GRAPH_CACHE_DIR': str(GRAPH_CACHE_DIR),
        'SPLIT_DIR': str(MEDIUM_SPLIT_DIR),
        'PYTHON_BIN': str(PYTHON_BIN),
        'FRAMEWORK': FRAMEWORK,
        'ASPECTS': ' '.join(ASPECTS),
        'MIN_TERM_FREQUENCY': str(MIN_TERM_FREQUENCY),
        'EPOCHS': str(MEDIUM_EPOCHS),
        'BATCH_SIZE': str(MEDIUM_BATCH_SIZE),
        'NUM_WORKERS': str(MEDIUM_NUM_WORKERS),
        'HIDDEN_DIM': str(MEDIUM_HIDDEN_DIM),
        'METRIC_THRESHOLD': str(METRIC_THRESHOLD),
        'FMAX_THRESHOLD_STEP': str(FMAX_THRESHOLD_STEP),
        'PROGRESS_MODE': PROGRESS_MODE,
        'PROGRESS_EVERY': str(PROGRESS_EVERY),
        'DEVICE': DEVICE,
        'SEED': str(SEED),
        'RUN_NAME': RUN_NAME,
        'RUN_DIR': str(RUN_DIR),
        'IA_FILE': str(IA_FILE),
        'GO_OBO_FILE': str(GO_OBO_FILE),
        'GROUND_TRUTH_FILE': str(GROUND_TRUTH_FILE),
        'REQUIRE_IA_FILE': '1',
        'CONTINUE_ON_FAILURE': '1',
    }
)

missing_eval_inputs = [str(path) for path in (IA_FILE,) if not path.exists()]
if missing_eval_inputs and SUBMIT_JOB and not DRY_RUN:
    raise FileNotFoundError(f'Missing required evaluation input(s): {missing_eval_inputs}')

submit_cmd = ['sbatch', str(TRAIN_SCRIPT)]
print('[submit env]')
for key in ['RUN_NAME', 'RUN_DIR', 'SPLIT_DIR', 'ASPECTS', 'EPOCHS', 'BATCH_SIZE', 'PROGRESS_EVERY', 'IA_FILE']:
    print(f'{key}={submit_env[key]}')
print('[submit cmd]', format_cmd(submit_cmd))

job_id = None
submit_output = None
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'notebook_submit_config.json').write_text(json.dumps(config | {'submit_env_subset': {k: submit_env[k] for k in ['RUN_NAME', 'RUN_DIR', 'SPLIT_DIR', 'ASPECTS', 'EPOCHS']}}, indent=2), encoding='utf-8')

if not SUBMIT_JOB:
    print('[skip] CAFA5_SUBMIT_MEDIUM_TRAINING is false')
elif DRY_RUN:
    print('[dry-run] job not submitted')
else:
    completed = subprocess.run(submit_cmd, env=submit_env, check=True, text=True, capture_output=True)
    submit_output = (completed.stdout + completed.stderr).strip()
    print(submit_output)
    match = re.search(r'Submitted batch job (\d+)', submit_output)
    if not match:
        raise RuntimeError(f'Could not parse Slurm job id from: {submit_output}')
    job_id = match.group(1)
    (RUN_DIR / 'slurm_job_id.txt').write_text(job_id + '\n', encoding='utf-8')
    print('job_id =', job_id)

## Monitor Job and Logs

Run these cells while the job is active. After the job completes, continue to the result-loading cells.

In [ ]:
# Check Slurm queue status for this job/user.
known_job_id_path = RUN_DIR / 'slurm_job_id.txt'
known_job_id = job_id if 'job_id' in globals() and job_id else (known_job_id_path.read_text(encoding='utf-8').strip() if known_job_id_path.exists() else None)

if known_job_id:
    cmd = ['squeue', '-j', known_job_id]
else:
    cmd = ['squeue', '-u', SERVER_USER]
print('[run]', format_cmd(cmd))
subprocess.run(cmd, check=False)
print('run_dir =', RUN_DIR)

In [ ]:
# Tail driver and per-aspect logs after the job has started.
def print_tail(path: Path, lines: int = 80) -> None:
    print(f'\n===== {path} =====')
    if not path.exists():
        print('[missing]')
        return
    text_lines = path.read_text(encoding='utf-8', errors='replace').splitlines()
    print('\n'.join(text_lines[-lines:]))

print_tail(RUN_DIR / 'driver.log', lines=120)
for aspect in ASPECTS:
    print_tail(RUN_DIR / aspect.lower() / 'train.log', lines=80)

## Load Results

Continue here after the Slurm job finishes and `results_summary.json` exists.

In [ ]:
# Load final result summaries.
results_summary_path = RUN_DIR / 'results_summary.json'
if not results_summary_path.exists():
    raise FileNotFoundError(f'Results summary is not ready yet: {results_summary_path}')

result_rows = json.loads(results_summary_path.read_text(encoding='utf-8'))
if pd is None:
    print(json.dumps(result_rows, indent=2))
else:
    results_df = pd.DataFrame(result_rows)
    display(results_df)
    print('results_summary_tsv =', RUN_DIR / 'results_summary.tsv')

In [ ]:
# Load per-epoch metrics from each aspect summary.json.
def load_epoch_rows(run_dir: Path, aspects: list[str]) -> list[dict]:
    rows = []
    for aspect in aspects:
        summary_path = run_dir / aspect.lower() / 'summary.json'
        if not summary_path.exists():
            print(f'[missing] {summary_path}')
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        for record in summary.get('history', []):
            for split_name in ('train', 'val', 'test'):
                metrics = record.get(split_name, {})
                row = {
                    'aspect': aspect,
                    'split': split_name,
                    'epoch': record.get('epoch'),
                    'epoch_seconds': record.get('epoch_seconds'),
                }
                for key in (
                    'loss',
                    'micro_precision',
                    'micro_recall',
                    'micro_f1',
                    'macro_precision',
                    'macro_recall',
                    'macro_f1',
                    'macro_f1_positive_labels',
                    'fmax',
                    'fmax_threshold',
                    'fmax_precision',
                    'fmax_recall',
                    'graphs',
                ):
                    row[key] = metrics.get(key)
                rows.append(row)
    return rows

epoch_rows = load_epoch_rows(RUN_DIR, ASPECTS)
if pd is None:
    print(json.dumps(epoch_rows[:20], indent=2))
else:
    epoch_df = pd.DataFrame(epoch_rows)
    display(epoch_df.head(20))
    print('rows =', len(epoch_df))

## Plots

The plots below use the unweighted metrics emitted by `train_minimal_graph_model.py`. CAFA IA-weighted `wFmax` still requires prediction TSV export plus `cafaeval`.

In [ ]:
# Plot per-epoch loss and F1/Fmax curves.
if pd is None or plt is None:
    raise ImportError('pandas and matplotlib are required for plotting')
if epoch_df.empty:
    raise ValueError('epoch_df is empty')

plot_splits = ['train', 'val', 'test']
metrics_to_plot = ['loss', 'micro_f1', 'macro_f1', 'fmax']
fig, axes = plt.subplots(len(metrics_to_plot), len(ASPECTS), figsize=(5 * len(ASPECTS), 3.2 * len(metrics_to_plot)), squeeze=False)

for column_index, aspect in enumerate(ASPECTS):
    aspect_df = epoch_df[epoch_df['aspect'] == aspect]
    for row_index, metric in enumerate(metrics_to_plot):
        ax = axes[row_index][column_index]
        for split_name in plot_splits:
            split_df = aspect_df[aspect_df['split'] == split_name].sort_values('epoch')
            if split_df.empty:
                continue
            ax.plot(split_df['epoch'], split_df[metric], marker='o', label=split_name)
        ax.set_title(f'{aspect} {metric}')
        ax.set_xlabel('epoch')
        ax.set_ylabel(metric)
        ax.grid(True, alpha=0.3)
        if row_index == 0 and column_index == 0:
            ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Plot final validation/test comparison by aspect.
if pd is None or plt is None:
    raise ImportError('pandas and matplotlib are required for plotting')

final_df = epoch_df.sort_values('epoch').groupby(['aspect', 'split'], as_index=False).tail(1)
metric_cols = ['micro_f1', 'macro_f1', 'fmax']
plot_df = final_df[final_df['split'].isin(['val', 'test'])][['aspect', 'split', *metric_cols]]
display(plot_df)

fig, axes = plt.subplots(1, len(metric_cols), figsize=(5 * len(metric_cols), 4), squeeze=False)
for axis, metric in zip(axes[0], metric_cols):
    pivot = plot_df.pivot(index='aspect', columns='split', values=metric).reindex(ASPECTS)
    pivot.plot(kind='bar', ax=axis)
    axis.set_title(f'Final {metric}')
    axis.set_xlabel('aspect')
    axis.set_ylabel(metric)
    axis.grid(True, axis='y', alpha=0.3)
    axis.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Result Notes

Record the actual observations here after the job finishes:

- Did all aspects complete successfully?
- Which aspect has the best validation Fmax?
- Is train/val divergence visible?
- Is the medium run stable enough to move to the full split?
- Next step: export prediction TSV and run CAFA evaluator with `eval_inputs/IA.txt`.